# Application Yamada regression: `Latest_Workplace` vs optimized branch

This notebook is a **plot-free reproduction of the Yamada calculations that actually appear in the application notebooks**. It does not use synthetic benchmark graphs as a substitute for those applications.

It reproduces:

- the 12 physics Yamada-table cases from `applications/01_physics_applications.ipynb`;
- the embedded K4 two-backend check from `applications/02_mathematics_applications.ipynb`;
- the mathematics $\Theta_s$ scan for $s=2,\ldots,7$;
- every `NOTEBOOK_YAMADA_EXAMPLES` structured-catalog case; and
- the `Cylinder(2,n)` scan for $n=3,\ldots,6$.

For each branch the same regression driver performs graph construction/extraction, projection selection where applicable, PD generation, and Yamada evaluation. The notebook then requires **literal equality of the complete result records**. No plotting or visualization code is executed.

## 1. Verify that this notebook is using the optimized checkout

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src' / 'knotted_graph').exists():
    raise RuntimeError('Run this notebook from inside the KnottedGraph repository checkout.')

SRC = ROOT / 'src'
sys.path.insert(0, str(SRC))
branch = subprocess.check_output(
    ['git', 'rev-parse', '--abbrev-ref', 'HEAD'], cwd=ROOT, text=True
).strip()
commit = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True
).strip()
print('ROOT   =', ROOT)
print('branch =', branch)
print('commit =', commit)
if branch != 'perf/yamada-max-optimization':
    raise RuntimeError(
        'Run this notebook from the perf/yamada-max-optimization branch; '
        f'current branch is {branch!r}.'
    )

import knotted_graph
kg_path = Path(knotted_graph.__file__).resolve()
print('knotted_graph loaded from =', kg_path)
if SRC not in kg_path.parents:
    raise RuntimeError('A stale installed knotted_graph was imported instead of this checkout.')


## 2. Run the exact same application-Yamada driver on both branches

The driver itself comes from this optimization checkout. Only `PYTHONPATH` and the working directory change, so both branches are tested with the **same case definitions**.

In [ ]:
DRIVER = ROOT / 'dev' / 'application_yamada_regression.py'
if not DRIVER.exists():
    raise FileNotFoundError(DRIVER)

def run_ref(ref):
    with tempfile.TemporaryDirectory() as td:
        work = Path(td) / 'repo'
        add = subprocess.run(
            ['git', 'worktree', 'add', '--detach', str(work), ref],
            cwd=ROOT, text=True, capture_output=True
        )
        if add.returncode:
            raise RuntimeError(
                f'Could not create worktree for {ref}:\n{add.stdout}\n{add.stderr}'
            )
        try:
            env = dict(os.environ)
            env['PYTHONPATH'] = str(work / 'src')
            env['PYTHONNOUSERSITE'] = '1'
            proc = subprocess.run(
                [sys.executable, str(DRIVER)],
                cwd=work, env=env, text=True, capture_output=True
            )
            if proc.returncode:
                raise RuntimeError(
                    f'Application Yamada regression failed for {ref} '
                    f'(exit {proc.returncode}).\n'
                    f'STDOUT:\n{proc.stdout}\n'
                    f'STDERR:\n{proc.stderr}'
                )
            return json.loads(proc.stdout)
        finally:
            subprocess.run(
                ['git', 'worktree', 'remove', '--force', str(work)],
                cwd=ROOT, text=True, capture_output=True, check=False
            )

print('Running Latest_Workplace application cases ...')
baseline = run_ref('Latest_Workplace')
print('Running perf/yamada-max-optimization application cases ...')
optimized = run_ref('perf/yamada-max-optimization')
print('baseline records  =', len(baseline))
print('optimized records =', len(optimized))


## 3. Exact branch-to-branch comparison

In [ ]:
if len(baseline) != len(optimized):
    raise AssertionError(
        f'Record-count mismatch: Latest_Workplace={len(baseline)}, optimized={len(optimized)}'
    )

differences = []
for index, (old, new) in enumerate(zip(baseline, optimized)):
    if old != new:
        differences.append((index, old, new))

if differences:
    print(f'FAIL: {len(differences)} application records changed.')
    for index, old, new in differences[:10]:
        print('\n--- first differing record index', index, '---')
        print('Latest_Workplace:')
        print(json.dumps(old, indent=2, sort_keys=True))
        print('optimized:')
        print(json.dumps(new, indent=2, sort_keys=True))
    raise AssertionError('Application-level Yamada output changed between branches.')

print('PASS: every reproduced application Yamada record is exactly unchanged.')


## 4. Inspect the reproduced application results

These are calculation records only: graph diagnostics, chosen rotation/crossing/PD data where applicable, and the Yamada polynomial.

In [ ]:
physics = [row for row in optimized if row['application'] == 'physics']
mathematics = [row for row in optimized if row['application'] == 'mathematics']

print(f'Physics application cases: {len(physics)}')
for row in physics:
    print(
        f"{row['case']:14s} gamma={row['gamma']:<4} "
        f"V={row['nodes']:<3} E={row['edges']:<3} crossings={row['crossings']:<2} "
        f"Yamada={row['yamada']}"
    )

print(f'\nMathematics application cases: {len(mathematics)}')
for row in mathematics:
    polynomial = row.get('yamada', row.get('yamada_negami'))
    print(f"{row['case']}: {polynomial}")


## Acceptance criterion

A PASS means that, for the Yamada-producing application cases reproduced here, the optimized branch has the same graph diagnostics and the same exact Yamada output as `Latest_Workplace`; for embedded cases it additionally requires the same selected rotation, crossing count and PD code. Because this notebook intentionally excludes plotting, it isolates the scientific Yamada calculations from visualization concerns.